# 01 - Historical NFL Data Collection

## Goal

This notebook collects every dataset required to build the NFL Season Projection Model.

Rather than downloading spreadsheets manually, every dataset is collected programmatically so the project can be reproduced and updated in future seasons.

## Data Sources

- nflverse
- NFL schedules
- Team rosters
- Coaching information
- Play-by-play data
- Team statistics
- Advanced metrics

## Output

Processed datasets are saved into the `/data` directory for use throughout the project.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

## Project Directory

In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

ROSTER_DIR = DATA_DIR / "roster"

COACHING_DIR = DATA_DIR / "coaching"

SCHEDULE_DIR = DATA_DIR / "schedules"

print(PROJECT_ROOT)

c:\Users\efriedman\Desktop\NFL-Season-Projections


In [8]:
import nfl_data_py as nfl

print("NFL data package loaded")

NFL data package loaded


## Historical Seasons

The first version of the model uses NFL data from 2015 through 2025.

This provides enough recent history to identify patterns while keeping the data relevant to the modern NFL. Older seasons may be added later for specific coaching, roster, and rule-change analysis.

In [9]:
SEASONS = list(range(2015, 2026))

print(SEASONS)
print(f"Number of seasons: {len(SEASONS)}")

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of seasons: 11


In [10]:
import nflreadpy as nfl

print("nflreadpy loaded")

nflreadpy loaded


## Test Play-by-Play Download

Before collecting the full historical dataset, one season is downloaded to confirm the data source, notebook environment, and file structure are working correctly.

In [11]:
pbp_test = nfl.load_pbp(seasons=[2025])

print(type(pbp_test))
print(pbp_test.shape)

<class 'polars.dataframe.frame.DataFrame'>
(48771, 372)


In [12]:
# Number of rows and columns
print(f"Rows: {pbp_test.height:,}")
print(f"Columns: {pbp_test.width}")

print("\nFirst 25 columns:\n")

print(pbp_test.columns[:25])

Rows: 48,771
Columns: 372

First 25 columns:

['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln']


In [13]:
pbp_test.head()

play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,…,home_coach,away_coach,stadium_id,game_stadium,aborted_play,success,passer,passer_jersey_number,rusher,rusher_jersey_number,receiver,receiver_jersey_number,pass,rush,first_down,special,play,passer_id,rusher_id,receiver_id,name,jersey_number,id,fantasy_player_name,fantasy_player_id,fantasy,fantasy_id,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
f64,str,str,str,str,str,i32,str,str,str,str,f64,str,f64,f64,f64,str,f64,f64,f64,f64,f64,i32,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,…,str,str,str,str,f64,f64,str,i32,str,i32,str,i32,f64,f64,f64,f64,f64,str,str,str,str,i32,str,str,str,str,str,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64
1.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,null,null,null,null,null,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,null,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,null,"""GAME""",null,null,0.0,0.0,null,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0.0,0.0,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.0,null,null,null,null,null,null,null
40.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""NO""",35.0,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,1.0,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,2.0,"""19-B.Grupe kicks 65 yards from…","""kickoff""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0.0,0.0,0.0,1.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.3527,null,null,null,null,null,null,null
63.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",78.0,"""2025-09-07""",896.0,1796.0,3596.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""14:56""","""ARI 22""",10.0,2.0,"""(14:56) 6-J.Conner right tackl…","""run""",3.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,"""J.Conner""",6,null,null,0.0,1.0,0.0,0.0,1.0,null,"""00-0033553""",null,"""J.Conner""",6,"""00-0033553""","""J.Conner""","""00-0033553""","""J.Conner""","""00-0033553""",0.0,0.0,-0.190052,null,null,null,null,null,0.511128,-51.112807
85.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",75.0,"""2025-09-07""",858.0,1758.0,3558.0,"""Half1""",0.0,1.0,0.0,1.0,2.0,0,"""14:18""","""ARI 25""",7.0,2.0,"""(14:18) (Shotgun) 1-K.Murray p…","""pass""",11.0,1.0,0.0,1.0,0.0,0.0,0.0,"""short""",…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,1.0,"""K.Murray""",1,null,null,"""T.McBride""",85,1.0,0.0,1.0,0.0,1.0,"""00-0035228""",null,"""00-0037744""","""K.Murray""",1,"""00-0035228""","""T.McBride""","""00-0037744""","""T.McBride""","""00-0037744""",1.0,0.0,1.31734,0.939998,4.750889,3,0.666726,0.43911,0.66894,33.105969
115.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",64.0,"""2025-09-07""",820.0,1720.0,3520.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""13:40""","""ARI 36""",10.0,2.0,"""(13:40) 1-K.Murray sacked at A…","""pass""",-11.0,0.0,0.0,1.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,"""K.Murray""",1,null,null,null,null,1.0,0.0,0.0,0.0,1.0,"""00-0035228""",null,null,"""K.Murray""",1,"""00-0035228""",null,null,n

# Data Collection Roadmap

The season projection model combines multiple datasets that each answer different football questions.

Rather than relying on a single dataset, the project integrates team performance, player performance, coaching continuity, roster movement, and schedule information into one unified modeling pipeline.

## Planned Datasets

### 1. Play-by-Play Data
Purpose:
- Calculate offensive and defensive efficiency
- EPA
- Success Rate
- Explosive Plays
- Turnovers
- Pressure
- Red Zone Performance

### 2. Team Season Statistics
Purpose:
- Wins
- Losses
- Point Differential
- Home/Road Performance
- One Score Games

### 3. Team Rosters
Purpose:
- Returning starters
- Position groups
- Player continuity

### 4. Player Statistics
Purpose:
- Quarterback projections
- Position group ratings
- Individual improvement

### 5. Coaching Database
Purpose:
- HC/OC/DC changes
- Coordinator continuity
- Previous performance

### 6. Schedule Data
Purpose:
- Opponents
- Home/Away
- Rest
- Travel
- Divisional games

### 7. Transactions
Purpose:
- Free agent additions
- Departures
- Trades
- Draft picks

## Save Raw Play-by-Play Data

The raw dataset is saved locally so future notebooks can use the same source data without downloading it again.

Parquet format is used because it is faster and more space-efficient than CSV while preserving data types.

In [14]:
test_output_path = RAW_DIR / "play_by_play_2025.parquet"

pbp_test.write_parquet(test_output_path)

print(f"Saved to: {test_output_path}")

Saved to: c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\play_by_play_2025.parquet


## Download Historical Play-by-Play Data

The full historical dataset is collected for the 2015–2025 seasons.

Each season is saved as its own Parquet file so the pipeline remains organized, easier to update, and less memory-intensive than combining everything immediately.

In [17]:
for season in SEASONS:
    output_path = RAW_DIR / f"play_by_play_{season}.parquet"

    if output_path.exists():
        print(f"{season}: file already exists — skipping")
        continue

    print(f"{season}: downloading...")

    season_pbp = nfl.load_pbp(seasons=[season])
    season_pbp.write_parquet(output_path)

    print(
        f"{season}: saved {season_pbp.height:,} rows "
        f"and {season_pbp.width} columns"
    )

2015: downloading...
2015: saved 48,122 rows and 372 columns
2016: downloading...
2016: saved 47,651 rows and 372 columns
2017: downloading...
2017: saved 47,245 rows and 372 columns
2018: downloading...
2018: saved 47,109 rows and 372 columns
2019: downloading...
2019: saved 47,260 rows and 372 columns
2020: downloading...
2020: saved 47,705 rows and 372 columns
2021: downloading...
2021: saved 49,922 rows and 372 columns
2022: downloading...
2022: saved 49,434 rows and 372 columns
2023: downloading...
2023: saved 49,665 rows and 372 columns
2024: downloading...
2024: saved 49,492 rows and 372 columns
2025: file already exists — skipping
